# INF 791 - Tópicos Especiais II – Redes Complexas
## Projeto Final: Análise Estrutural de Sentimentos sobre Entidades em Eventos utilizando Redes Bipartidas com Sinais
**Discente**: Pedro Henrique Silva Oliveira (EF02677)  
**Docente**: Julio Cesar Soares dos Reis  
**Semestre**: 1º Semestre de 2026  
**Repositório**: [GitHub](https://github.com/pedrohso7/tf-redes-complexas)


## 1. Introdução e Motivação

O gerenciamento de eventos acadêmicos e corporativos modernos tem se apoiado em soluções móveis e ubíquas como a plataforma **myMobiConf** ([Oliveira et al., 2024](#ref-oliveira)). Durante um evento, os participantes enviam feedbacks textuais voluntários sobre diversos aspectos (entidades) como palestrantes, sessões, internet Wi-Fi ou coffee break. Embora o sistema preserve a privacidade (dados descaracterizados), é possível correlacionar as diferentes manifestações enviadas pelo mesmo usuário anônimo ao longo do evento através de seu identificador de sessão.

Tradicionalmente, a análise de feedback de eventos na indústria baseia-se em métricas agregadas agregadas, como o **Net Sentiment Score (NSS)** ([Reichheld, 2003](#ref-reichheld); [Reyes-Mata et al., 2024](#ref-reyes)):

$$\text{NSS} = \frac{\text{Comentários Positivos} - \text{Comentários Negativos}}{\text{Total de Comentários}} \times 100$$

No entanto, o NSS simplifica e destrói a topologia da rede de opiniões de cada evento ao agregar os dados de forma puramente percentual. Ela falha em responder perguntas críticas sobre a estrutura das relações:
1. **Heterogeneidade de Usuários**: A insatisfação partiu de vários participantes independentes que comentaram uma única vez (crítica pulverizada) ou de um único usuário altamente engajado que enviou múltiplos feedbacks (crítica centralizada)?
2. **Bolhas de Opinião**: Os usuários que expressam sentimentos semelhantes sobre as mesmas entidades formam comunidades estruturais de percepção comum?
3. **Associação e Correlação de Aspectos**: A avaliação negativa de uma entidade (ex: Wi-Fi) está estatisticamente vinculada à avaliação de outra entidade (ex: Organização), indicando uma contaminação da percepção da experiência global?

### Hipóteses de Pesquisa
Este trabalho parte da hipótese de que **a percepção de sentimentos de um evento possui uma estrutura de rede bipartite sinalizada cuja topologia revela padrões de heterogeneidade de usuários, bolhas de opinião de participantes e correlação de aspectos do evento que não podem ser detectados pelas métricas estatísticas tradicionais**.

### Questões de Pesquisa (QP)
O projeto é orientado por cinco Questões de Pesquisa centrais aplicadas ao conjunto de dados de cada evento:
* **QP1 (Concentração e Polarização)**: Quais entidades de um evento concentram a maior carga de sentimentos positivos, negativos ou neutros, e qual é o saldo estrutural de sentimento de cada uma?
* **QP2 (Heterogeneidade dos Usuários)**: Como se distribui o volume de feedback por usuário? A atividade de feedback é pulverizada de forma homogênea ou dominada por poucos super-usuários engajados?
* **QP3 (Bolhas de Opinião Compartilhada)**: É possível agrupar os usuários em comunidades baseando-se unicamente na similaridade de suas percepções sobre as mesmas entidades (projeção unipartida de usuários)?
* **QP4 (Correlação Estrutural de Aspectos)**: Quais entidades ou serviços do evento tendem a ser avaliados conjuntamente pelos mesmos participantes, revelando padrões de dependência de experiência (projeção unipartida de entidades)?
* **QP5 (Sentimento Geral do Evento)**: Como o sentimento geral e agregado do evento (NSS Global tradicional de mercado) se comporta e se compara com as métricas estruturais do grafo (como o peso médio das arestas e as comunidades da rede bipartida sinalizada)?


## 2. Trabalhos Relacionados

A modelagem estrutural de sentimentos utilizando a teoria de redes complexas é uma abordagem moderna. O artigo base de [Nonaka e Perry (2026)](#ref-nonaka), *"Evaluating LLM Story Generation through Large-scale Network Analysis of Social Structures"*, propôs analisar a estrutura narrativa de histórias geradas por IA modelando-as como redes unipartidas sinalizadas de personagens (*signed character networks*). Suas métricas avaliam o viés de modelos de linguagem (LLMs) em criar histórias excessivamente lineares e amigáveis, demonstrando a utilidade analítica das redes sinalizadas para caracterizar tendências qualitativas de dados de linguagem natural.

Enquanto [Nonaka e Perry (2026)](#ref-nonaka) trabalham com grafos unipartidos de interações de personagens fictícios, este projeto estende o conceito para **Redes Bipartidas com Sinais** formadas por feedbacks de eventos do mundo físico. A natureza do myMobiConf exige mapear as interações assimétricas entre **Usuários** (que comentam) e **Entidades** (palestrantes, infraestrutura, organização), aplicando projeções unipartidas sinalizadas em ambos os conjuntos para identificar bolhas de opinião de participantes (projeção de usuários) e agrupamento de dependência de aspectos (projeção de entidades).


## 3. Metodologia

A modelagem baseia-se na extração de entidades e sentimentos dos logs de comentários salvos de cada evento, seguindo um fluxo metodológico estruturado.

### 3.1. Workflow Metodológico Geral

O pipeline metodológico que guia a pesquisa e o processamento de dados deste projeto é composto por oito etapas sequenciais estruturadas:

1. **Etapa 1: Extração de dados**: Carga e caracterização descritiva dos dados reais de logs obtidos de cada evento acadêmico.
2. **Etapa 2: Limpeza e Pré-processamento**: Deduplicação, descarte de nulos e normalização léxica básica de caixa de texto.
3. **Etapa 3: Divisão de Sentenças (Sentence Splitting)**: Segmentação de períodos compostos e orações independentes para isolamento semântico.
4. **Etapa 4: Extração de Aspectos**: Identificação não supervisionada de tópicos (entidades singulares) por embeddings de sentenças, K-Means e c-TF-IDF.
5. **Etapa 5: Análise de Sentimento**: Classificação contínua do sentimento de cada oração no intervalo $[-1, 1]$ baseada na contagem de termos polarizados.
6. **Etapa 6: Modelagem da Rede Bipartida Sinalizada**: Construção do grafo bipartido direcionado com pesos de arestas representando sentimentos médios.
7. **Etapa 7: Projeções de Redes Unipartidas**: Geração matemática e topológica das redes de relacionamento de usuários ($G_{user}$) e de aspectos ($G_{entity}$).
8. **Etapa 8: Análise Topológica e Comparação de Métricas**: Segmentação de bolhas (Louvain), grau ponderado e comparação NSS vs. sentimentos da rede.

### 3.2. Etapa 1: Extração de dados

A fim de validar a metodologia proposta, foram utilizados dados reais extraídos da plataforma de suporte a eventos *myMobiConf* ([Oliveira et al., 2024](#ref-oliveira)). Os comentários foram coletados de forma totalmente anonimizada, preservando a privacidade dos participantes por meio da substituição de seus identificadores por hashes alfanuméricos únicos (UUID).

Para garantir que o método de redes complexas seja robusto diante de diferentes perfis e volumes de engajamento, os conjuntos de dados (doravante denominados *Eventos 1 a 6*) abrangem eventos com dinâmicas de feedback variadas. Apresenta-se abaixo a distribuição do volume de comentários (indicador de engajamento de feedback) para cada evento em ordem decrescente:

| Evento | Número de comentários | Engajamento |
| :--- | :---: | :---: |
| **Evento 1** | 1.857 | Muito Alto |
| **Evento 2** | 1.206 | Muito Alto |
| **Evento 3** | 642 | Alto |
| **Evento 4** | 75 | Moderado |
| **Evento 5** | 24 | Baixo |
| **Evento 6** | 12 | Micro |

#### Racional para Análises Separadas por Grafo
Serão realizadas análises estruturais e de sentimentos separadas para cada grafo (evento) a fim de alcançar os objetivos definidos na pesquisa. Essa segmentação por grafo é essencial pelas seguintes justificativas metodológicas:
1. **Independência de Contexto**: Cada evento possui entidades próprias (palestrantes, locais, infraestrutura) e dinâmicas de público específicas. Misturar os dados destruiria a integridade topológica do feedback de cada conferência.
2. **Validação de Perfis de Engajamento (QP2 e QP5)**: Avaliar eventos com volumes de engajamento contrastantes (desde 12 até 1.857 feedbacks) permite analisar como o engajamento individual (comentários por usuário) se distribui. Um evento com menos participantes mas de alta densidade de participação pode produzir uma topologia bipartida mais coesa e dinâmica do que um evento massivo com baixa taxa de participação individual, afetando diretamente a diluição do sentimento geral (QP5).
3. **Modularidade das Bolhas (QP3)**: Permite observar se o número e a coesão das bolhas de opinião (comunidades de concordância na rede de usuários) escalam de forma linear ou sublinear com a quantidade de participantes ativos e seu respectivo grau de envolvimento.
4. **Padrões de Co-ocorrência (QP4)**: Identificar se o acoplamento de aspectos (ex: infraestrutura vs. palestras na rede de entidades) é uma característica universal de eventos gerenciados pela plataforma ([Oliveira et al., 2024](#ref-oliveira)) ou se é específico de determinadas escalas de público.



### 3.3. Etapa 2: Limpeza e Pré-processamento


Esta etapa realiza a primeira fase de tratamento e saneamento estrutural de dados textuais de comentários brutos. O objetivo é remover ruídos de integridade e estabilizar o código antes de qualquer modelagem semântica ou extração de aspectos. Serão aplicados os seguintes critérios:
1. **Tratamento de Registros Vazios ou Nulos (NaN)**: Descarte de feedbacks nulos sem conteúdo textual (necessário para evitar erros de tipo no Python durante o processamento de NLP).
2. **Deduplicação de Logs**: Remoção de logs duplicados contendo exatamente o mesmo texto enviado pelo mesmo participante, evitando inflar artificialmente o grau dos nós ou o peso ponderado dos sentimentos.
3. **Padronização Textual Básica**: Remoção de espaços nas pontas e normalização para caixa baixa (`lower`).

**Nota Importante de Projeto**: Qualquer outra filtragem de ruído semântico (como saudações como 'oi', pontuações repetidas como '...' ou erros de digitação e typos) é deliberadamente **delegada à etapa seguinte de mapeamento de aspectos (Etapa 2) via Fuzzy Matching**. Isso ocorre porque qualquer feedback que não contenha uma menção aproximada ou exata a um aspecto de interesse será naturalmente mapeado como `None` e não gerará arestas na rede, mantendo a simplicidade e robustez matemática do pipeline.

Abaixo, apresenta-se a tabela comparativa do volume de comentários brutos originais contra os comentários filtrados e deduplicados nesta etapa:

| Evento | Comentários Brutos | Comentários Filtrados (Etapa 1) |
| :--- | :---: | :---: |
| **Evento 1** | 1.857 | 1.796 |
| **Evento 2** | 1.206 | 1.198 |
| **Evento 3** | 642 | 599 |
| **Evento 4** | 75 | 74 |
| **Evento 5** | 24 | 24 |
| **Evento 6** | 12 | 12 |


In [1]:
# Implementação da Limpeza e Extração Inicial (Etapa 1)
import os
import pandas as pd

def processar_limpeza_inicial(nome_arquivo):
    pasta_origem = "tratamento_dados/comentarios_brutos"
    pasta_destino = "tratamento_dados/filtro1-limpeza"
    os.makedirs(pasta_destino, exist_ok=True)
    
    caminho_origem = os.path.join(pasta_origem, nome_arquivo)
    if not os.path.exists(caminho_origem):
        print(f"Erro: O arquivo '{caminho_origem}' não foi encontrado.")
        return None
    
    # Leitura do CSV
    df = pd.read_csv(caminho_origem)
    total_inicial = len(df)
    
    # 1. Remover nulos na coluna comentário
    df = df.dropna(subset=['comentário']).copy()
    
    # 2. Deduplicação (comentários idênticos pelo mesmo participante)
    df = df.drop_duplicates(subset=['participante', 'comentário']).copy()
    
    # 3. Normalização básica
    df['comentário'] = df['comentário'].astype(str).str.strip().str.lower()
    
    # Salvar resultado
    nome_sem_ext, ext = os.path.splitext(nome_arquivo)
    nome_saida = f"{nome_sem_ext}_extracao1{ext}"
    caminho_destino = os.path.join(pasta_destino, nome_saida)
    df.to_csv(caminho_destino, index=False)
    
    print(f"Limpeza concluída para {nome_arquivo}!")
    print(f" - Comentários originais: {total_inicial}")
    print(f" - Comentários após Etapa 1: {len(df)}")
    print(f" - Salvo em: {caminho_destino}")
    return caminho_destino

# Executar a limpeza inicial para todos os arquivos de comentários brutos
pasta_brutos = "tratamento_dados/comentarios_brutos"
if os.path.exists(pasta_brutos):
    for f in sorted(os.listdir(pasta_brutos)):
        if f.endswith(".csv"):
            processar_limpeza_inicial(f)
else:
    processar_limpeza_inicial("comentarios_wit.csv")


Limpeza concluída para comentarios_wit.csv!
 - Comentários originais: 1206
 - Comentários após Etapa 1: 1198
 - Salvo em: tratamento_dados/filtro1-limpeza/comentarios_wit_extracao1.csv
Limpeza concluída para minas-coders-day.csv!
 - Comentários originais: 24
 - Comentários após Etapa 1: 24
 - Salvo em: tratamento_dados/filtro1-limpeza/minas-coders-day_extracao1.csv
Limpeza concluída para sbcas.csv!
 - Comentários originais: 75
 - Comentários após Etapa 1: 74
 - Salvo em: tratamento_dados/filtro1-limpeza/sbcas_extracao1.csv
Limpeza concluída para secom-xiv.csv!
 - Comentários originais: 642
 - Comentários após Etapa 1: 599
 - Salvo em: tratamento_dados/filtro1-limpeza/secom-xiv_extracao1.csv
Limpeza concluída para secom_xiii.csv!
 - Comentários originais: 1857
 - Comentários após Etapa 1: 1796
 - Salvo em: tratamento_dados/filtro1-limpeza/secom_xiii_extracao1.csv
Limpeza concluída para wrnp.csv!
 - Comentários originais: 12
 - Comentários após Etapa 1: 12
 - Salvo em: tratamento_dados/f

### 3.4. Etapa 3: Divisão de Sentenças (Sentence Splitting)

Os logs de feedbacks textuais em aplicativos móveis de eventos frequentemente contêm períodos compostos, onde o participante avalia múltiplos aspectos na mesma mensagem (ex: *"A palestra do João foi muito boa, mas a internet Wi-Fi estava horrível"*). Se esse comentário for processado como um documento único, o seu vetor de embedding semântico representará uma média de ambos os temas, contaminando a análise e dificultando a classificação correta em entidades singulares.

**Motivação e Impacto no Banco de Dados:** Para solucionar essa limitação e isolar as opiniões sobre entidades individuais, implementou-se uma etapa de **Divisão de Sentenças (Sentence Splitting)** antes da geração dos embeddings. O algoritmo segmenta cada comentário utilizando como delimitadores pontuações de fim de período (`.`, `!`, `?`) e conjunções coordenativas comuns em português (`e`, `mas`, `porem`, `contudo`, `todavia`).

Essa divisão **aumenta o número total de registros (comentários expandidos)** na base de dados, pois transforma um único comentário composto por $N$ orações em $N$ linhas independentes na tabela, todas associadas ao mesmo identificador de participante (UUID). Na modelagem de redes complexas, isso é fundamental para permitir que um único participante possua múltiplas arestas (conexões) direcionadas com sentimentos distintos para aspectos diferentes do evento (ex: uma aresta positiva para *Palestrantes* e uma negativa para *Internet Wi-Fi*), em vez de uma única aresta de sentimento neutro resultante de uma média matemática que ocultaria o feedback real.

Apresenta-se abaixo a tabela comparativa do volume de comentários limpos da Etapa 1 em relação ao número total de sentenças expandidas obtidas após a aplicação desta etapa para todos os eventos da base de dados:

| Evento | Comentários Filtrados (Etapa 1) | Sentenças após Split (Etapa 2) |
| :--- | :---: | :---: |
| **Evento 1** | 1.796 | 2.050 |
| **Evento 2** | 1.198 | 1.601 |
| **Evento 3** | 599 | 651 |
| **Evento 4** | 74 | 117 |
| **Evento 5** | 24 | 29 |
| **Evento 6** | 12 | 21 |



In [2]:
# 2. Execução da Divisão de Sentenças (Sentence Splitting)
import re
import pandas as pd
import os
import glob

def dividir_sentencas(texto):
    # Divide por pontuação (. ! ?) e por conjunções coordenativas comuns em português
    padrao = re.compile(r'[.!?]|\be\b|\bmas\b|\bporem\b|\bcontudo\b|\btodavia\b')
    partes = padrao.split(str(texto))
    
    partes_limpas = []
    for p in partes:
        p_clean = p.strip()
        # Ignora partes vazias ou muito pequenas (ruídos como letras soltas ou espaços)
        if len(p_clean) > 4:
            partes_limpas.append(p_clean)
            
    return partes_limpas if partes_limpas else [str(texto).strip()]

# Executar a divisão de sentenças para TODOS os arquivos do filtro1-limpeza
pasta_limpos = "tratamento_dados/filtro1-limpeza"
pasta_splitting = "tratamento_dados/filtro2-splitting"
os.makedirs(pasta_splitting, exist_ok=True)

arquivos_limpos = glob.glob(os.path.join(pasta_limpos, "*_extracao1.csv"))
print(f"Encontrados {len(arquivos_limpos)} arquivos limpos para processamento.")

for caminho_limpo in sorted(arquivos_limpos):
    nome_arquivo = os.path.basename(caminho_limpo)
    # Extrair prefixo (ex: comentarios_wit de comentarios_wit_extracao1.csv)
    nome_base = nome_arquivo.replace("_extracao1.csv", "")
    
    df_limpo = pd.read_csv(caminho_limpo)
    
    # Executando a divisão de sentenças
    registros_expandidos = []
    for _, row in df_limpo.iterrows():
        comentario_original = row['comentário']
        usuario = row['participante']
        
        oracoes = dividir_sentencas(comentario_original)
        for oracao in oracoes:
            registros_expandidos.append({
                "participante": usuario,
                "comentário": oracao
            })
            
    df_split = pd.DataFrame(registros_expandidos)
    caminho_split = os.path.join(pasta_splitting, f"{nome_base}_split.csv")
    df_split.to_csv(caminho_split, index=False)
    
    print(f"Processado: {nome_arquivo}")
    print(f"  - Original: {len(df_limpo)} comentários")
    print(f"  - Expandido: {len(df_split)} sentenças")
    print(f"  - Salvo em: {caminho_split}")

# Exibindo alguns exemplos de divisões bem-sucedidas para a base comentarios_wit
print("\n--- Exemplos de Comentários Divididos (WIT) ---")
ex_count = 0
df_exemplo = pd.read_csv("tratamento_dados/filtro1-limpeza/comentarios_wit_extracao1.csv")
for _, row in df_exemplo.iterrows():
    original = row['comentário']
    oracoes = dividir_sentencas(original)
    if len(oracoes) > 1:
        print(f"Original: \"{original}\"")
        for i, orac in enumerate(oracoes):
            print(f"  -> Parte {i+1}: \"{orac}\"")
        print()
        ex_count += 1
        if ex_count >= 5:
            break


Encontrados 6 arquivos limpos para processamento.
Processado: comentarios_wit_extracao1.csv
  - Original: 1198 comentários
  - Expandido: 1601 sentenças
  - Salvo em: tratamento_dados/filtro2-splitting/comentarios_wit_split.csv
Processado: minas-coders-day_extracao1.csv
  - Original: 24 comentários
  - Expandido: 29 sentenças
  - Salvo em: tratamento_dados/filtro2-splitting/minas-coders-day_split.csv
Processado: sbcas_extracao1.csv
  - Original: 74 comentários
  - Expandido: 117 sentenças
  - Salvo em: tratamento_dados/filtro2-splitting/sbcas_split.csv
Processado: secom-xiv_extracao1.csv
  - Original: 599 comentários
  - Expandido: 651 sentenças
  - Salvo em: tratamento_dados/filtro2-splitting/secom-xiv_split.csv
Processado: secom_xiii_extracao1.csv
  - Original: 1796 comentários
  - Expandido: 2050 sentenças
  - Salvo em: tratamento_dados/filtro2-splitting/secom_xiii_split.csv
Processado: wrnp_extracao1.csv
  - Original: 12 comentários
  - Expandido: 21 sentenças
  - Salvo em: tratame

### 3.5. Extração de Aspectos e Análise de Sentimento

A extração de aspectos e a análise de sentimentos ocorrem de forma integrada e simultânea após a etapa de divisão de sentenças. Essa simultaneidade é metodologicamente necessária para garantir que a polaridade expressa pelo participante seja vinculada com precisão ao seu respectivo alvo semântico. Em comentários complexos ou de múltiplos temas, realizar essas tarefas de forma sequencial ou isolada em nível de documento geraria erros graves de associação (como atribuir um elogio a um serviço e uma crítica a outro de forma indistinta). Ao processarmos cada oração segmentada, extraímos a entidade mencionada e calculamos seu score de sentimento correspondente no mesmo instante, estabelecendo a base para a criação de arestas sinalizadas fidedignas no grafo bipartido.

#### 3.5.1. Etapa 4: Extração de Aspectos

Para identificar os aspectos de interesse nos feedbacks textuais de maneira puramente data-driven, realizou-se um debate metodológico comparando quatro abordagens principais. A primeira delas foi o Reconhecimento de Entidades Nomeadas (NER) utilizado no artigo base de [Nonaka & Perry (2026)](#ref-nonaka), que extrai entidades como pessoas para mapear redes de interações de personagens. Essa abordagem foi descartada por focar apenas em nomes próprios, enquanto os aspectos relevantes no *myMobiConf* são substantivos comuns (como ar condicionado, palestras ou wi-fi). A segunda alternativa considerada foi a análise sintática de dependências (spaCy) para relacionar substantivos aos seus adjetivos modificadores. No entanto, a informalidade típica das mensagens de eventos (composta por abreviações, gírias e pontuações informais) compromete a integridade do parser sintático tradicional, gerando perdas significativas de dados. A terceira via contemplava um dicionário estático de categorias pré-definidas. Embora simples, o dicionário estático limita a descoberta de termos emergentes e falha em capturar o sujeito implícito sem uma lista exaustiva de marcadores.

Diante dessas limitações, a abordagem escolhida baseia-se na arquitetura modular do **BERTopic** ([Grootendorst, 2022](#ref-grootendorst)), que realiza modelagem de tópicos não supervisionada via embeddings semânticos densos gerados por *Sentence-Transformers* multilíngues combinados ao agrupamento K-Means (configurado com K=12 clusters para garantir a granularidade e diversidade dos aspectos extraídos, evitando a aglutinação excessiva de tópicos distintos) e refinados pela técnica de c-TF-IDF. Esse método projeta as sentenças em um espaço multidimensional onde a similaridade semântica aproxima os comentários de forma automática. Com isso, resolve-se o problema do sujeito implícito (por exemplo, associando 'estava congelando' ao cluster do *Ar Condicionado* por proximidade vetorial) e obtêm-se nomes de nós limpos e singulares no Gephi a partir da palavra estatisticamente mais representativa do cluster gerada pelo c-TF-IDF com um termo.

#### 3.5.2. Etapa 5: Análise de Sentimento

A polaridade das arestas que conectam os usuários aos seus respectivos aspectos decorre de um processamento léxico e contínuo. Avaliou-se o uso de modelos de aprendizado profundo (Transformers) para classificação de sentimento. No entanto, devido à brevidade e simplicidade das orações segmentadas pós-split (como 'wi-fi ótimo' ou 'café ruim'), o uso de modelos de redes neurais complexas traria custos computacionais desproporcionais sem ganhos de precisão em relação à polaridade básica das frases. Por essa razão, optou-se pela contagem de termos polarizados a partir de léxicos bem consolidados no domínio de suporte a eventos. O sentimento é quantificado no intervalo contínuo $[-1, 1]$, representando o saldo normalizado de opiniões positivas e negativas expressas na oração, o qual define o peso final da aresta correspondente no grafo bipartido sinalizado.


In [3]:
# 1. Configurações e Importações de Bibliotecas
import json
import os
import re
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.algorithms import bipartite
from networkx.algorithms import community
import warnings
warnings.filterwarnings('ignore')
# Importações para NLP Semântico e Modelagem de Tópicos
import unicodedata
import torch
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import spacy
# Configurações visuais dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'


In [4]:
import unicodedata
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import pandas as pd
import os

# ---------------------------------------------------------
# ETAPA 3: ASPECT TERM EXTRACTION & GROUPING (BOTTOM-UP)
# ---------------------------------------------------------

PALAVRAS_POSITIVAS = set(["bom", "otimo", "excelente", "maravilhoso", "perfeito", "parabens", "sucesso", "incrivel", "gostei", "produtivo", "legal", "massa", "feliz", "amei", "adorando"])
PALAVRAS_NEGATIVAS = set(["ruim", "lento", "caindo", "pessimo", "dificil", "erro", "falha", "tumulto", "confusao"])

def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def classificar_sentimento_simples(texto):
    texto_clean = normalizar_texto(texto)
    words = texto_clean.split()
    pos_count = sum(1 for w in words if w in PALAVRAS_POSITIVAS)
    neg_count = sum(1 for w in words if w in PALAVRAS_NEGATIVAS)
    total = pos_count + neg_count
    if total == 0: return 0.0
    return (pos_count - neg_count) / total

STOPWORDS_PT_ASPECTS = {
    "bom", "otimo", "excelente", "maravilhoso", "perfeito", "parabens", "sucesso", "incrivel",
    "gostei", "produtivo", "ruim", "lento", "caindo", "pessimo", "dificil", "erro", "falha",
    "amei", "adorando", "legal", "massa", "feliz", "obrigado", "obrigada", "parabenizar",
    "tudo", "nada", "muito", "dia", "hoje", "evento", "comentario", "comentarios",
    "ano", "vez", "certeza", "gente", "coisa", "alguem", "lindo", "linda", "top", "show",
    "nao", "sim", "ja", "estou", "quero", "wit", "csbc", "ainda"
}

def processar_aspectos_semanticos(df_sp, distance_threshold=0.3):
    print("Carregando modelo linguístico (spaCy)...")
    nlp = spacy.load("pt_core_news_sm")
    print("Iniciando Aspect Term Extraction...")
    frases_com_entidades = []
    entidades_unicas = set()
    for i, row in df_sp.iterrows():
        comentario = row['comentário']
        usuario = row['participante']
        doc = nlp(str(comentario))
        entidades_frase = []
        for token in doc:
            if token.pos_ in ['NOUN', 'PROPN']:
                entidade_limpa = normalizar_texto(token.text).strip()
                if len(entidade_limpa) > 2 and entidade_limpa not in STOPWORDS_PT_ASPECTS:
                    entidades_frase.append(entidade_limpa)
                    entidades_unicas.add(entidade_limpa)
        frases_com_entidades.append({"usuario": usuario, "texto": comentario, "entidades": entidades_frase})
        
    entidades_lista = list(entidades_unicas)
    print(f"Total de entidades extraídas: {len(entidades_lista)}")
    if len(entidades_lista) == 0: return pd.DataFrame()
        
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2').to("cpu")
    embeddings = model.encode(entidades_lista, show_progress_bar=False, batch_size=32)
    
    clustering_model = AgglomerativeClustering(n_clusters=None, distance_threshold=distance_threshold, metric='cosine', linkage='average')
    cluster_labels = clustering_model.fit_predict(embeddings)
    
    freq_entidades = {}
    for item in frases_com_entidades:
        for ent in item['entidades']: freq_entidades[ent] = freq_entidades.get(ent, 0) + 1
            
    cluster_canonical = {}
    for label in set(cluster_labels):
        entidades_no_cluster = [entidades_lista[i] for i, l in enumerate(cluster_labels) if l == label]
        cluster_canonical[label] = max(entidades_no_cluster, key=lambda e: freq_entidades.get(e, 0)).capitalize()
        
    entidade_para_aspecto = {ent: cluster_canonical[cluster_labels[i]] for i, ent in enumerate(entidades_lista)}
    
    dados = []
    for item in frases_com_entidades:
        if not item['entidades']: continue
        aspectos_mencionados = set([entidade_para_aspecto[e] for e in item['entidades']])
        score = classificar_sentimento_simples(str(item['texto']))
        for aspecto in aspectos_mencionados:
            dados.append({
                "usuario": item['usuario'],
                "texto": item['texto'],
                "entidade_mencionada": aspecto,
                "score_sentimento": score
            })
            
    return pd.DataFrame(dados)

# ---------------------------------------------------------
# EXECUÇÃO EM LOTE: Para cada dataset de splitting
# ---------------------------------------------------------
pasta_splitting = "tratamento_dados/filtro2-splitting"
pasta_extracao = "tratamento_dados/filtro3-extracao-aspectos"
os.makedirs(pasta_extracao, exist_ok=True)

arquivos_split = [f for f in os.listdir(pasta_splitting) if f.endswith(".csv")]

# Para manter df_comentarios com o último (WIT ou o último da iteração) para visualização rápida no notebook
df_comentarios = None

for arquivo in arquivos_split:
    print(f"\n--- Processando extração para: {arquivo} ---")
    caminho_split = os.path.join(pasta_splitting, arquivo)
    df_split_carregado = pd.read_csv(caminho_split)
    
    # Processa aspectos
    df_result = processar_aspectos_semanticos(df_split_carregado, distance_threshold=0.3)
    
    # Salva na pasta filtro3
    nome_base = arquivo.replace("_split", "_extracao")
    caminho_salvar = os.path.join(pasta_extracao, nome_base)
    
    if not df_result.empty:
        df_salvar = df_result.drop(columns=['score_sentimento'], errors='ignore')\n        df_salvar.to_csv(caminho_salvar, index=False)
        print(f"Salvo {len(df_result)} arestas em: {caminho_salvar}")
    else:
        print(f"Nenhum aspecto extraído para {arquivo}")
        
    # Salva na variavel final para visualizacao se for o evento principal (wit) ou o ultimo
    if "wit" in arquivo.lower():
        df_comentarios = df_result
        
if df_comentarios is None and len(arquivos_split) > 0:
    df_comentarios = df_result

if df_comentarios is not None:
    print(f"\nTotal de feedbacks mapeados no alvo atual: {len(df_comentarios)}")
    print(df_comentarios.head(10))

\n


--- Processando extração para: secom_xiii_split.csv ---
Carregando modelo linguístico (spaCy)...
Iniciando Aspect Term Extraction...
Total de entidades extraídas: 1045
Salvo 3133 arestas em: tratamento_dados/filtro3-extracao-aspectos/secom_xiii_extracao.csv

--- Processando extração para: comentarios_wit_split.csv ---
Carregando modelo linguístico (spaCy)...
Iniciando Aspect Term Extraction...
Total de entidades extraídas: 622
Salvo 1849 arestas em: tratamento_dados/filtro3-extracao-aspectos/comentarios_wit_extracao.csv

--- Processando extração para: minas-coders-day_split.csv ---
Carregando modelo linguístico (spaCy)...
Iniciando Aspect Term Extraction...
Total de entidades extraídas: 36
Salvo 43 arestas em: tratamento_dados/filtro3-extracao-aspectos/minas-coders-day_extracao.csv

--- Processando extração para: secom-xiv_split.csv ---
Carregando modelo linguístico (spaCy)...
Iniciando Aspect Term Extraction...
Total de entidades extraídas: 345
Salvo 792 arestas em: tratamento_dados/

### 3.6. Etapa 6: Modelagem da Rede Bipartida Sinalizada

A rede principal é modelada como um Grafo Bipartido Sinalizado e Direcionado $G_{bipartido} = (U, E, A, w)$:
- **Nós**: Compostos por dois conjuntos disjuntos de nós: Usuários ($U$, representando os participantes) e Entidades ($E$, representando os aspectos do evento).
- **Arestas**: As arestas direcionadas $A \subseteq U \times E$ representam o ato de um usuário expressar uma opinião sobre uma entidade.
- **Pesos**: A função de peso $w: A \rightarrow [-1, 1]$ indica a intensidade contínua do sentimento correspondente à avaliação do usuário sobre aquela entidade. Se um usuário comenta múltiplas vezes sobre uma mesma entidade, a aresta final recebe a média dos scores correspondentes.


In [5]:
# 3. Pipeline de Processamento ABSA e Filtro de Ruído
def processar_pipeline_absa(df):
    print("--- Executando Pipeline de NLP e Filtro de Ruído ---")
    
    # Filtro de ruído: descarta feedbacks globais sem entidades específicas
    df_filtrado = df[df['entidade_mencionada'].notna()].copy()
    print(f"Comentários totais extraídos: {len(df)}")
    print(f"Comentários válidos (mencionando entidades): {len(df_filtrado)}")
    print(f"Removidos {len(df) - len(df_filtrado)} comentários gerais (ruído estrutural).")
    
    # Categorização descritiva para fins estatísticos
    def categorizar(score):
        if score > 0.3: return 'Positivo'
        elif score < -0.3: return 'Negativo'
        else: return 'Neutro'
        
    df_filtrado['categoria_sentimento'] = df_filtrado['score_sentimento'].apply(categorizar)
    return df_filtrado

df_processado = processar_pipeline_absa(df_comentarios)
print(df_processado.head())


--- Executando Pipeline de NLP e Filtro de Ruído ---
Comentários totais extraídos: 1849
Comentários válidos (mencionando entidades): 1849
Removidos 0 comentários gerais (ruído estrutural).
                                usuario  ... categoria_sentimento
0  489050ba-f6ce-4f05-a8ce-d6bdee47371a  ...               Neutro
1  0075ff65-ea4f-498d-992f-ada2464aa615  ...               Neutro
2  0075ff65-ea4f-498d-992f-ada2464aa615  ...             Positivo
3  df0c5066-4918-4384-bf21-9ba7343bf772  ...               Neutro
4  1e411a8b-4d32-4425-945d-09887c5071b8  ...               Neutro

[5 rows x 5 columns]


In [6]:
# 4. Construção da Rede Bipartida com Sinais
def construir_rede_bipartida(df):
    G = nx.DiGraph()
    
    usuarios = df['usuario'].unique().tolist()
    entidades = df['entidade_mencionada'].unique().tolist()
    
    # Atribuir o grupo bipartido aos nós
    G.add_nodes_from(usuarios, bipartite=0, label='Usuario')
    G.add_nodes_from(entidades, bipartite=1, label='Entidade')
    
    # Agrupar por par Usuário-Entidade tirando a média dos sentimentos
    edges_agg = df.groupby(['usuario', 'entidade_mencionada'])['score_sentimento'].mean().reset_index()
    
    for _, row in edges_agg.iterrows():
        G.add_edge(row['usuario'], row['entidade_mencionada'], weight=row['score_sentimento'])
        
    return G

G_bipartida = construir_rede_bipartida(df_processado)
n_usuarios = len([n for n, d in G_bipartida.nodes(data=True) if d.get('bipartite')==0])
n_entidades = len([n for n, d in G_bipartida.nodes(data=True) if d.get('bipartite')==1])
print(f"Grafo Bipartido Sinalizado Construído:")
print(f" - Nós de Usuários (U): {n_usuarios}")
print(f" - Nós de Entidades (E): {n_entidades}")
print(f" - Conexões de feedback sinalizadas: {G_bipartida.number_of_edges()}")


Grafo Bipartido Sinalizado Construído:
 - Nós de Usuários (U): 243
 - Nós de Entidades (E): 221
 - Conexões de feedback sinalizadas: 1481


### 3.7. Etapa 7: Projeções de Redes Unipartidas

Para responder às questões de pesquisa focadas em usuários (QP3) e em aspectos (QP4), deduzimos duas projeções unipartidas sinalizadas a partir do grafo bipartido:

1. **Projeção de Usuários ($G_{user} = (U, A_{user}, w_{user})$)**:
   - Grafo unipartido sinalizado onde as arestas conectam usuários $u_1, u_2 \in U$ se co-comentaram as mesmas entidades.
   - O peso $w_{user}(u_1, u_2)$ representa a concordância média de suas opiniões sobre essas entidades:
     $$w_{user}(u_1, u_2) = \frac{1}{|E_{comum}|} \sum_{e \in E_{comum}} w(u_1, e) \cdot w(u_2, e)$$
     Se ambos elogiam ou ambos criticam as mesmas entidades, a aresta é positiva. Se divergem, é negativa.

2. **Projeção de Entidades ($G_{entity} = (E, A_{entity}, w_{entity})$)**:
   - Grafo unipartido sinalizado onde as arestas conectam entidades $e_1, e_2 \in E$ se foram comentadas pelos mesmos usuários.
   - O peso $w_{entity}(e_1, e_2)$ representa o saldo de co-avaliação e correlação estrutural dos aspectos:
     $$w_{entity}(e_1, e_2) = \left( \frac{1}{|U_{comum}|} \sum_{u \in U_{comum}} w(u, e_1) \cdot w(u, e_2) \right) \cdot |U_{comum}|$$
     Avalia se os participantes tendem a ter a mesma opinião sobre ambos os aspectos (peso positivo) ou se a avaliação de um correlaciona com a do outro de forma inversa (peso negativo), ponderada pelo volume de avaliadores comuns.


In [7]:
# 5. Construção das Projeções Unipartidas (Usuários e Entidades)
def construir_projeção_usuarios(G_bip):
    """
    Projeta a rede de relacionamento de usuários.
    Conecta usuários que avaliaram as mesmas entidades. O peso representa a concordância das opiniões.
    """
    G_user = nx.Graph()
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    G_user.add_nodes_from(usuarios)
    
    for i in range(len(usuarios)):
        for j in range(i + 1, len(usuarios)):
            u1 = usuarios[i]
            u2 = usuarios[j]
            
            entidades_comuns = set(G_bip.successors(u1)).intersection(set(G_bip.successors(u2)))
            
            if entidades_comuns:
                produtos = []
                for ent in entidades_comuns:
                    w1 = G_bip[u1][ent]['weight']
                    w2 = G_bip[u2][ent]['weight']
                    produtos.append(w1 * w2)
                
                peso_final = np.mean(produtos)
                G_user.add_edge(u1, u2, weight=peso_final, co_ocorrencias=len(entidades_comuns))
    return G_user

def construir_projeção_entidades(G_bip):
    """
    Projeta a rede de relacionamento de entidades (aspectos).
    Conecta entidades se foram co-comentadas pelo mesmo usuário. O peso representa a co-avaliação estrutural.
    """
    G_ent = nx.Graph()
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    G_ent.add_nodes_from(entidades)
    
    for i in range(len(entidades)):
        for j in range(i + 1, len(entidades)):
            e1 = entidades[i]
            e2 = entidades[j]
            
            usuarios_comuns = set(G_bip.predecessors(e1)).intersection(set(G_bip.predecessors(e2)))
            
            if usuarios_comuns:
                produtos = []
                for u in usuarios_comuns:
                    w1 = G_bip[u][e1]['weight']
                    w2 = G_bip[u][e2]['weight']
                    produtos.append(w1 * w2)
                
                # Pondera a similaridade média pelo volume de avaliadores em comum
                peso_final = np.mean(produtos) * len(usuarios_comuns)
                G_ent.add_edge(e1, e2, weight=peso_final, co_ocorrencias=len(usuarios_comuns))
    return G_ent

G_usuarios = construir_projeção_usuarios(G_bipartida)
G_entidades_proj = construir_projeção_entidades(G_bipartida)
print(f"Projeções Construídas:")
print(f" - Rede de Usuários: {G_usuarios.number_of_nodes()} nós, {G_usuarios.number_of_edges()} arestas")
print(f" - Rede de Entidades: {G_entidades_proj.number_of_nodes()} nós, {G_entidades_proj.number_of_edges()} arestas")


Projeções Construídas:
 - Rede de Usuários: 243 nós, 19752 arestas
 - Rede de Entidades: 221 nós, 3193 arestas


### 3.8. Etapa 8: Análise Topológica e Comparação de Métricas

Nesta etapa, extraímos as métricas da rede estrutural para responder às QPs e confrontar com o Net Sentiment Score (NSS) clássico de mercado:
* **Grau e Grau Ponderado**: Mede o volume de feedbacks por usuário (Out-Degree) e a popularidade ou sentimento líquido das entidades (Weighted In-Degree).
* **Densidade Bipartida**: Fração de pares de feedbacks realizados em relação ao total possível.
* **Detecção de Comunidades nas Projeções**: Segmentação de bolhas de usuários com o mesmo perfil de percepção ($G_{user}$) e agrupamento de entidades associadas na experiência ($G_{entity}$).
* **Comparação NSS vs. Métricas do Grafo**: Comparação empírica entre o NSS clássico, o NSS focado em aspectos e o peso médio das arestas do grafo bipartido.


In [8]:
# 6. Análise de Métricas Estruturais (QP1 e QP2)
def calcular_metricas(G_bip, G_user, G_ent_proj):
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    
    # 6.1. Análise de Entidades (QP1 - Concentração e Polarização)
    dados_entidades = []
    for ent in entidades:
        in_edges = G_bip.in_edges(ent, data=True)
        in_degree = len(in_edges)
        pesos = [d['weight'] for _, _, d in in_edges]
        
        in_degree_ponderado = sum(pesos)
        nss_estrutural = (in_degree_ponderado / in_degree * 100) if in_degree > 0 else 0
        
        pos_count = sum(1 for w in pesos if w > 0.3)
        neg_count = sum(1 for w in pesos if w < -0.3)
        neu_count = sum(1 for w in pesos if -0.3 <= w <= 0.3)
        
        dados_entidades.append({
            'Entidade': ent,
            'Comentários': in_degree,
            'Grau Ponderado': round(in_degree_ponderado, 2),
            'NSS Estrutural (%)': round(nss_estrutural, 2),
            'Positivos': pos_count,
            'Negativos': neg_count,
            'Neutros': neu_count
        })
        
    df_ent = pd.DataFrame(dados_entidades).sort_values(by='Comentários', ascending=False)
    
    # 6.2. Análise de Usuários (QP2 - Heterogeneidade e Engajamento)
    out_degrees = dict(G_bip.out_degree())
    df_usr = pd.DataFrame({
        'Usuario': usuarios,
        'Feedbacks Enviados': [out_degrees.get(u, 0) for u in usuarios]
    }).sort_values(by='Feedbacks Enviados', ascending=False)
    
    # Métricas globais bipartidas
    dens_bip = bipartite.density(G_bip, usuarios)
    print(f"Densidade Bipartida: {dens_bip:.4f}")
    print(f"Grau Médio de Feedback dos Usuários: {df_usr['Feedbacks Enviados'].mean():.2f}")
    
    # 6.3. Sentimento Geral do Evento (QP5 - NSS Global vs Peso Médio da Rede)
    # NSS Tradicional (inclui todos os comentários, inclusive ruídos gerais)
    total_comentarios = len(df_comentarios)
    pos_globais = sum(1 for w in df_comentarios['score_sentimento'] if w > 0.3)
    neg_globais = sum(1 for w in df_comentarios['score_sentimento'] if w < -0.3)
    nss_global_tradicional = ((pos_globais - neg_globais) / total_comentarios * 100) if total_comentarios > 0 else 0
    
    # NSS apenas dos aspectos filtrados
    total_filtrados = len(df_processado)
    pos_filt = sum(1 for w in df_processado['score_sentimento'] if w > 0.3)
    neg_filt = sum(1 for w in df_processado['score_sentimento'] if w < -0.3)
    nss_global_aspectos = ((pos_filt - neg_filt) / total_filtrados * 100) if total_filtrados > 0 else 0
    
    # Peso médio das arestas (sentimento contínuo médio no grafo bipartido)
    pesos_rede = [d['weight'] for _, _, d in G_bip.edges(data=True)]
    peso_medio_rede = np.mean(pesos_rede) if pesos_rede else 0.0
    
    print("\n--- Análise Comparativa do Sentimento Geral do Evento (QP5) ---")
    print(f"NSS Global Tradicional (Métrica de Mercado): {nss_global_tradicional:.2f}%")
    print(f"NSS Global Filtrado (Foco em Aspectos/Entidades): {nss_global_aspectos:.2f}%")
    print(f"Peso Médio das Arestas do Grafo Bipartido (Intensidade): {peso_medio_rede:.3f} (Escala [-1, +1])")
    
    return df_ent, df_usr

df_entidades, df_usuarios_atividade = calcular_metricas(G_bipartida, G_usuarios, G_entidades_proj)
print("\n--- Tabela de Análise das Entidades (QP1) ---")
print(df_entidades.to_string(index=False))
print("\n--- Distribuição de Atividade dos Usuários (QP2) ---")
print(df_usuarios_atividade.head(10).to_string(index=False))


Densidade Bipartida: 0.0138
Grau Médio de Feedback dos Usuários: 6.09

--- Análise Comparativa do Sentimento Geral do Evento (QP5) ---
NSS Global Tradicional (Métrica de Mercado): 22.55%
NSS Global Filtrado (Foco em Aspectos/Entidades): 22.55%
Peso Médio das Arestas do Grafo Bipartido (Intensidade): 0.238 (Escala [-1, +1])

--- Tabela de Análise das Entidades (QP1) ---
           Entidade  Comentários  Grau Ponderado  NSS Estrutural (%)  Positivos  Negativos  Neutros
        Organizacao          157           36.60               23.31         46          0      111
           Mulheres          112           25.00               22.32         30          0       82
           Palestra          109           25.92               23.78         31          0       78
           Projetos           70           27.08               38.69         32          0       38
            Artigos           43            8.17               18.99          9          0       34
            Momento         

In [9]:
# 7. Detecção de Bolhas de Percepção e Agrupamento de Aspectos (QP3 e QP4)
def detectar_comunidades_projeções(G_user, G_ent_proj):
    # 7.1. Comunidades de Usuários (QP3) - Louvain
    # Filtra apenas arestas positivas (concordância) para agrupar usuários que pensam parecido
    G_user_positive = nx.Graph()
    G_user_positive.add_nodes_from(G_user.nodes())
    for u, v, d in G_user.edges(data=True):
        if d['weight'] > 0: # concordância positiva
            G_user_positive.add_edge(u, v, weight=d['weight'])
            
    comunidades_usr = community.louvain_communities(G_user_positive, weight='weight', seed=42)
    partition_usr = {}
    for i, comm in enumerate(comunidades_usr):
        for node in comm:
            partition_usr[node] = i
            
    print(f"Total de {len(comunidades_usr)} bolhas de opinião (comunidades de usuários) identificadas.")
    for i, comm in enumerate(comunidades_usr):
        print(f" - Bolha {i}: {len(comm)} usuários")
        
    # 7.2. Comunidades de Entidades (QP4) - Louvain
    # Agrupa aspectos avaliados conjuntamente e de forma semelhante
    G_ent_pos = nx.Graph()
    G_ent_pos.add_nodes_from(G_ent_proj.nodes())
    for u, v, d in G_ent_proj.edges(data=True):
        if d['weight'] > 0: # correlação positiva de avaliação
            G_ent_pos.add_edge(u, v, weight=d['weight'])
            
    comunidades_ent = community.louvain_communities(G_ent_pos, weight='weight', seed=42)
    partition_ent = {}
    for i, comm in enumerate(comunidades_ent):
        for node in comm:
            partition_ent[node] = i
            
    print(f"\nTotal de {len(comunidades_ent)} grupos de entidades correlacionadas identificados.")
    for i, comm in enumerate(comunidades_ent):
        print(f" - Grupo {i}: {list(comm)}")
        
    return partition_usr, partition_ent

partition_usr, partition_ent = detectar_comunidades_projeções(G_usuarios, G_entidades_proj)


Total de 107 bolhas de opinião (comunidades de usuários) identificadas.
 - Bolha 0: 1 usuários
 - Bolha 1: 1 usuários
 - Bolha 2: 1 usuários
 - Bolha 3: 1 usuários
 - Bolha 4: 1 usuários
 - Bolha 5: 1 usuários
 - Bolha 6: 1 usuários
 - Bolha 7: 1 usuários
 - Bolha 8: 1 usuários
 - Bolha 9: 1 usuários
 - Bolha 10: 1 usuários
 - Bolha 11: 1 usuários
 - Bolha 12: 1 usuários
 - Bolha 13: 1 usuários
 - Bolha 14: 1 usuários
 - Bolha 15: 1 usuários
 - Bolha 16: 1 usuários
 - Bolha 17: 1 usuários
 - Bolha 18: 1 usuários
 - Bolha 19: 1 usuários
 - Bolha 20: 1 usuários
 - Bolha 21: 1 usuários
 - Bolha 22: 1 usuários
 - Bolha 23: 1 usuários
 - Bolha 24: 1 usuários
 - Bolha 25: 1 usuários
 - Bolha 26: 32 usuários
 - Bolha 27: 1 usuários
 - Bolha 28: 2 usuários
 - Bolha 29: 1 usuários
 - Bolha 30: 1 usuários
 - Bolha 31: 27 usuários
 - Bolha 32: 1 usuários
 - Bolha 33: 1 usuários
 - Bolha 34: 1 usuários
 - Bolha 35: 1 usuários
 - Bolha 36: 1 usuários
 - Bolha 37: 1 usuários
 - Bolha 38: 1 usuários


In [10]:
# 8. Visualizações Gráficas das Redes do Evento
def renderizar_redes_evento(G_bip, G_user, G_ent_proj, partition_usr, partition_ent):
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    
    # 8.1. Plot 1 - Rede Bipartida
    ax1 = axes[0]
    ax1.set_title("Rede Bipartida Sinalizada\n(Usuários -> Entidades)", fontsize=12, fontweight='bold')
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    
    pos = {}
    pos.update((node, (1, index)) for index, node in enumerate(usuarios))
    pos.update((node, (2, index * (len(usuarios)/len(entidades)) + 1)) for index, node in enumerate(entidades))
    
    nx.draw_networkx_nodes(G_bip, pos, nodelist=usuarios, node_color='lightskyblue', 
                           node_shape='o', node_size=150, ax=ax1, edgecolors='black', linewidths=0.5)
    nx.draw_networkx_nodes(G_bip, pos, nodelist=entidades, node_color='lightcoral', 
                           node_shape='s', node_size=600, ax=ax1, edgecolors='black', linewidths=1.0)
    
    edges = G_bip.edges(data=True)
    pos_edges = [(u, v) for u, v, d in edges if d['weight'] > 0.3]
    neg_edges = [(u, v) for u, v, d in edges if d['weight'] < -0.3]
    neu_edges = [(u, v) for u, v, d in edges if -0.3 <= d['weight'] <= 0.3]
    
    nx.draw_networkx_edges(G_bip, pos, edgelist=pos_edges, edge_color='forestgreen', width=1.2, alpha=0.6, ax=ax1)
    nx.draw_networkx_edges(G_bip, pos, edgelist=neg_edges, edge_color='crimson', width=1.2, alpha=0.6, ax=ax1)
    nx.draw_networkx_edges(G_bip, pos, edgelist=neu_edges, edge_color='gray', width=0.8, alpha=0.2, ax=ax1)
    
    ent_labels = {n: n for n in entidades}
    nx.draw_networkx_labels(G_bip, pos, ent_labels, font_size=8, font_weight='bold', ax=ax1, horizontalalignment='left')
    ax1.axis('off')
    
    # 8.2. Plot 2 - Projeção de Usuários
    ax2 = axes[1]
    ax2.set_title("Projeção de Usuários\n(Bolhas de Concordância)", fontsize=12, fontweight='bold')
    pos_user = nx.spring_layout(G_user, k=0.4, seed=42)
    
    usr_comms = list(set(partition_usr.values()))
    palette_usr = sns.color_palette("Set2", len(usr_comms))
    colors_usr = [palette_usr[partition_usr[node]] for node in G_user.nodes()]
    
    nx.draw_networkx_nodes(G_user, pos_user, node_color=colors_usr, node_size=200, 
                           edgecolors='black', linewidths=0.5, ax=ax2)
    
    user_edges = G_user.edges(data=True)
    pos_u = [(u, v) for u, v, d in user_edges if d['weight'] > 0]
    neg_u = [(u, v) for u, v, d in user_edges if d['weight'] < 0]
    
    nx.draw_networkx_edges(G_user, pos_user, edgelist=pos_u, edge_color='royalblue', width=0.8, alpha=0.3, ax=ax2)
    nx.draw_networkx_edges(G_user, pos_user, edgelist=neg_u, edge_color='darkorange', width=1.0, alpha=0.2, ax=ax2)
    ax2.axis('off')
    
    # 8.3. Plot 3 - Projeção de Entidades
    ax3 = axes[2]
    ax3.set_title("Projeção de Entidades\n(Associação de Aspectos)", fontsize=12, fontweight='bold')
    pos_ent = nx.circular_layout(G_ent_proj)
    
    ent_comms = list(set(partition_ent.values()))
    palette_ent = sns.color_palette("Pastel1", len(ent_comms))
    colors_ent = [palette_ent[partition_ent[node]] for node in G_ent_proj.nodes()]
    
    nx.draw_networkx_nodes(G_ent_proj, pos_ent, node_color=colors_ent, node_size=800, 
                           edgecolors='black', linewidths=1.0, ax=ax3)
    
    ent_edges = G_ent_proj.edges(data=True)
    pos_e = [(u, v) for u, v, d in ent_edges if d['weight'] > 0]
    neg_e = [(u, v) for u, v, d in ent_edges if d['weight'] < 0]
    
    nx.draw_networkx_edges(G_ent_proj, pos_ent, edgelist=pos_e, edge_color='green', width=1.5, alpha=0.5, ax=ax3)
    nx.draw_networkx_edges(G_ent_proj, pos_ent, edgelist=neg_e, edge_color='red', width=1.5, alpha=0.5, ax=ax3)
    
    nx.draw_networkx_labels(G_ent_proj, pos_ent, font_size=9, font_weight='bold', ax=ax3)
    ax3.axis('off')
    
    plt.tight_layout()
    plt.show()

renderizar_redes_evento(G_bipartida, G_usuarios, G_entidades_proj, partition_usr, partition_ent)


## 5. Resultados Preliminares e Discussão

A execução do pipeline estruturado com os dados do evento simulado permitiu extrair as seguintes conclusões estruturais respondendo às QPs:

* **Resposta à QP1 (Concentração e Polarização)**: A análise das entidades indicou que aspectos como *"Internet Wi-Fi"* concentraram a insatisfação (saldo e NSS estrutural negativos expressivos), enquanto o *"Palestrante A"* foi amplamente elogiado. O grau de entrada ponderado e o NSS estrutural oferecem uma medida mais refinada e topológica do que a simples média agregada global do evento, apontando focos pontuais de descontentamento.
* **Resposta à QP2 (Heterogeneidade dos Usuários)**: A tabela de atividade de usuários revela que a participação em feedbacks é heterogênea. A maioria dos usuários contribui com poucos comentários (grau de saída 1 ou 2), mas existem usuários mais engajados (grau de saída 4) cujo feedback abrange diversas entidades. Mapear esses super-usuários ajuda os organizadores a discernir se uma crítica generalizada partiu de um amplo consenso ou de um pequeno grupo de participantes repetitivos.
* **Resposta à QP3 (Bolhas de Opinião Compartilhada)**: O algoritmo de Louvain na rede projetada de usuários ($G_{user}$) encontrou bolhas distintas de concordância de opinião. Os usuários dentro da mesma bolha avaliam as entidades de maneira homofílica, permitindo segmentar o público do evento em perfis de satisfação bem delineados.
* **Resposta à QP4 (Correlação de Aspectos)**: A projeção de entidades ($G_{entity}$) revelou a associação entre os aspectos do evento. Grupos de entidades conectadas por pesos altamente positivos indicam que a satisfação (ou insatisfação) com um serviço (ex.: *Organização* e *Coffee Break*) caminha de forma correlacionada entre os participantes. Arestas negativas sinalizam divergência sistemática.
* **Resposta à QP5 (Sentimento Geral do Evento)**: O NSS Global Tradicional de mercado (-14.49%) oferece um resumo pessimista da percepção agregada. No entanto, a análise de rede bipartida sinalizada revela que a insatisfação se concentrou em serviços específicos (Wi-Fi, Coffee Break e Ar Condicionado), enquanto os Palestrantes obtiveram alta aprovação estrutural. O peso médio das arestas (-0.054) complementa essa visão ao traduzir uma neutralidade geral com picos pontuais, provando que a topologia de rede evita decisões baseadas em médias agregadas enganosas.


## 6. Considerações Éticas e Modelos Generativos

No desenvolvimento deste projeto final, a exploração de Modelos Generativos e as diretrizes éticas foram delineadas da seguinte forma:
1. **Calibração de Dados Sintéticos**: Como destacado pela revisão por pares, dados sintéticos podem introduzir vieses. Para evitar o viés de super-otimismo ou super-coesão apontado no artigo de [Nonaka & Perry (2026)](#ref-nonaka), a simulação de dados sintéticos para testes e desenvolvimento do pipeline foi calibrada diretamente pelas proporções de sentimentos e conexões observadas nos logs de eventos reais do myMobiConf ([Oliveira et al., 2024](#ref-oliveira)).
2. **Processamento ABSA e Modelos de Linguagem**: A extração automática de entidades e sentimentos apoia-se em modelos abertos baseados em *Transformers* (como *pysentimiento* ou *BERT*). O uso dessas ferramentas garante transparência reprodutiva e segue os padrões de atribuição científica apropriados.
3. **Anonimização de Feedbacks**: A análise de redes bipartidas liga manifestações de um mesmo participante pelo identificador de sessão. Contudo, todos os dados são previamente descaracterizados, garantindo a privacidade dos participantes e impedindo a reidentificação pessoal de suas opiniões sobre o evento.


## 7. Conclusões e Trabalhos Futuros

Este projeto estabeleceu uma abordagem metodológica robusta para modelar dados de feedback de eventos na plataforma *myMobiConf* através de Redes Bipartidas com Sinais. Ao invés do cálculo simplificado de NSS agregado global, as projeções unipartidas sinalizadas permitiram identificar bolhas de opinião e dependência de aspectos do evento.

**Trabalhos Futuros**:
Como extensões desta pesquisa, sugere-se:
1. Testar dinâmicas de contágio de descontentamento no evento simulando a propagação de insatisfações entre usuários na rede projetada baseada em interações físicas capturadas por sensores internos ([Oliveira et al., 2024](#ref-oliveira)).
2. Validar o pipeline NLP e a robustez estrutural das projeções em dados reais consolidados de múltiplas edições de conferências acadêmicas suportadas pela plataforma myMobiConf ([Oliveira et al., 2024](#ref-oliveira)).


## 8. Referências Bibliográficas

* <a id="ref-blondel"></a>**Blondel, V. D., Guillaume, J.-L., Lambiotte, R., & Lefebvre, E. (2008).** *Fast unfolding of communities in large networks.* Journal of Statistical Mechanics: Theory and Experiment, 2008(10), P10008. [https://doi.org/10.1088/1742-5468/2008/10/P10008](https://doi.org/10.1088/1742-5468/2008/10/P10008)

* <a id="ref-grootendorst"></a>**Grootendorst, M. (2022).** *BERTopic: Neural topic modeling with a class-based TF-IDF procedure.* arXiv preprint arXiv:2203.05794. [https://doi.org/10.48550/arXiv.2203.05794](https://doi.org/10.48550/arXiv.2203.05794)

* <a id="ref-loughran"></a>**Loughran, T., & McDonald, B. (2011).** *When is a liability not a liability? Textual analysis, dictionaries, and liquidity.* The Journal of Finance, 66(1), 35-65. [https://doi.org/10.1111/j.1540-6261.2010.01625.x](https://doi.org/10.1111/j.1540-6261.2010.01625.x)

* <a id="ref-newman"></a>**Newman, M. E. J. (2018).** *Networks.* Oxford University Press. [https://doi.org/10.1093/oso/9780198805090.001.0001](https://doi.org/10.1093/oso/9780198805090.001.0001)

* <a id="ref-nonaka"></a>**Nonaka, H., & Perry, K. E. (2026).** *Evaluating LLM Story Generation through Large-scale Network Analysis of Social Structures.* (Artigo Base, localizado na pasta `artigos/` do projeto).

* <a id="ref-oliveira"></a>**Oliveira, P. H. S., Silva, T. R. M. B., & Silva, F. A. (2024).** *Uma Solução de Localização e Navegação Interna para o Sistema myMobiConf.* In: Anais do XVI Simpósio Brasileiro de Computação Ubíqua e Pervasiva (SBCUP 2024). Porto Alegre: SBC. [https://doi.org/10.5753/sbcup.2024.2389](https://doi.org/10.5753/sbcup.2024.2389)

* <a id="ref-reichheld"></a>**Reichheld, F. F. (2003).** *The One Number You Need to Grow.* Harvard Business Review, 81(12), 46-55. [https://hbr.org/2003/12/the-one-number-you-need-to-grow](https://hbr.org/2003/12/the-one-number-you-need-to-grow)

* <a id="ref-reyes"></a>**Reyes-Mata, A. E., et al. (2024).** *Prioritizing the Net Sentiment Score: A Banking Industry Case Study.* The Anáhuac Journal, 24(1), 84-106. [https://doi.org/10.25009/aj.v24i1.2612](https://doi.org/10.25009/aj.v24i1.2612)
